# Enhanced Gold Layer: Healthcare-Style Analytics

## Purpose
This notebook creates 4 advanced Gold layer tables that mirror operational analytics patterns commonly used in healthcare executive dashboards and operational scorecards.

## Healthcare Analytics Parallels

### 1. Rolling KPIs (`mff_gold.rolling_kpis`)
**Healthcare Equivalent**: 
- **Census Rolling Averages**: 7-day/30-day rolling average daily census
- **Readmission Rolling Rates**: 30-day rolling readmission rates
- **HCAHPS Rolling Scores**: Rolling patient satisfaction scores
- **Safety Metrics**: Rolling infection rates, fall rates, pressure ulcer rates

**Business Value**: Smooth out daily volatility to identify true trends

### 2. Period-over-Period Comparisons (`mff_gold.period_over_period`)
**Healthcare Equivalent**:
- **WoW Hospital Metrics**: Current week vs prior week for operational dashboards
- **MoM Financial Performance**: Month-over-month revenue, volumes, margins
- **YoY Strategic Metrics**: Year-over-year growth for board reporting
- **Trending Indicators**: UP/DOWN/STABLE flags for executive scorecards

**Business Value**: Quickly identify performance changes requiring action

### 3. Executive Scorecard (`mff_gold.executive_scorecard`)
**Healthcare Equivalent**:
- **Balanced Scorecard**: Composite performance score across multiple dimensions
- **Quality Dashboard**: Weighted score from multiple quality measures
- **RAG Status**: Red/Amber/Green flags for executive reporting
- **CMS Star Ratings**: Composite performance scores from multiple measures

**Business Value**: Single metric summarizing overall organizational health

### 4. Channel Efficiency Matrix (`mff_gold.channel_efficiency_matrix`)
**Healthcare Equivalent**:
- **Payer Mix Analysis**: Efficiency metrics by insurance type (commercial, Medicare, Medicaid)
- **Referral Source Efficiency**: Which physicians/facilities send most profitable patients
- **Service Line Performance**: Ranked performance across cardiology, orthopedics, oncology
- **Clinic Productivity**: Comparative efficiency across clinic locations

**Business Value**: Resource allocation decisions based on efficiency rankings

## Implementation Approach
- All tables use window functions for rolling calculations
- LAG/LEAD functions for period-over-period comparisons
- CASE statements for RAG status determination
- RANK/DENSE_RANK for efficiency rankings
- Comprehensive documentation for self-service analytics

In [0]:
# =============================================================================
# TABLE 1: ROLLING KPIS
# =============================================================================
# Healthcare Parallel: 7-day/30-day/90-day rolling averages of key operational
# metrics - equivalent to census rolling averages, readmission rates, HCAHPS scores

print("\n" + "="*80)
print("CREATING: mff_gold.rolling_kpis")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TABLE mff_gold.rolling_kpis
    COMMENT 'Rolling KPI calculations over 7/30/90-day windows - equivalent to healthcare census and quality metric rolling averages'
    AS
    
    WITH daily_metrics AS (
        -- Calculate daily metrics from orders and sessions
        -- Healthcare equivalent: Daily census, admissions, discharges, procedures
        SELECT
            DATE(o.created_at) as metric_date,
            COUNT(DISTINCT o.order_id) as daily_orders,
            SUM(o.price_usd) as daily_revenue,
            COUNT(DISTINCT ws.website_session_id) as daily_sessions,
            COUNT(DISTINCT o.order_id) * 1.0 / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as daily_cvr,
            SUM(CASE WHEN oir.order_item_refund_id IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT o.order_id), 0) as daily_refund_rate
        FROM mff_bronze.website_sessions ws
        LEFT JOIN mff_bronze.orders o 
            ON ws.website_session_id = o.website_session_id
        LEFT JOIN mff_bronze.order_items oi
            ON o.order_id = oi.order_id
        LEFT JOIN mff_bronze.order_item_refunds oir
            ON oi.order_item_id = oir.order_item_id
        GROUP BY DATE(o.created_at)
    )
    
    SELECT
        metric_date,
        
        -- Daily actuals
        daily_orders,
        daily_revenue,
        daily_sessions,
        ROUND(daily_cvr, 4) as daily_cvr,
        ROUND(daily_refund_rate, 4) as daily_refund_rate,
        
        -- 7-day rolling averages
        -- Healthcare: Weekly operational metrics for frontline managers
        ROUND(AVG(daily_orders) OVER (
            ORDER BY metric_date 
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) as orders_7d_avg,
        
        ROUND(AVG(daily_revenue) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2) as revenue_7d_avg,
        
        ROUND(AVG(daily_cvr) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 4) as cvr_7d_avg,
        
        ROUND(AVG(daily_refund_rate) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 4) as refund_rate_7d_avg,
        
        -- 30-day rolling averages
        -- Healthcare: Monthly trending for department directors
        ROUND(AVG(daily_orders) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 2) as orders_30d_avg,
        
        ROUND(AVG(daily_revenue) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 2) as revenue_30d_avg,
        
        ROUND(AVG(daily_cvr) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 4) as cvr_30d_avg,
        
        ROUND(AVG(daily_refund_rate) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
        ), 4) as refund_rate_30d_avg,
        
        -- 90-day rolling averages
        -- Healthcare: Quarterly trending for executives and board
        ROUND(AVG(daily_orders) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ), 2) as orders_90d_avg,
        
        ROUND(AVG(daily_revenue) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ), 2) as revenue_90d_avg,
        
        ROUND(AVG(daily_cvr) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ), 4) as cvr_90d_avg,
        
        ROUND(AVG(daily_refund_rate) OVER (
            ORDER BY metric_date
            ROWS BETWEEN 89 PRECEDING AND CURRENT ROW
        ), 4) as refund_rate_90d_avg,
        
        CURRENT_TIMESTAMP() as created_at
        
    FROM daily_metrics
    WHERE metric_date IS NOT NULL
    ORDER BY metric_date
""")

# Verify table creation and show sample
result_count = spark.sql("SELECT COUNT(*) as cnt FROM mff_gold.rolling_kpis").collect()[0]['cnt']
print(f"\n✓ Created mff_gold.rolling_kpis with {result_count:,} rows")
print("\nSample data (most recent 5 days):")
spark.sql("""
    SELECT 
        metric_date,
        daily_orders,
        orders_7d_avg,
        orders_30d_avg,
        daily_revenue,
        revenue_30d_avg,
        daily_cvr,
        cvr_30d_avg
    FROM mff_gold.rolling_kpis
    ORDER BY metric_date DESC
    LIMIT 5
""").show(truncate=False)

print("\nHealthcare Use Case:")
print("  • Identify sustained trends vs daily volatility")
print("  • Smooth out weekend/holiday effects")
print("  • Early warning system for declining performance")
print("  • Benchmark current performance vs recent history")

In [0]:
# =============================================================================
# TABLE 2: PERIOD-OVER-PERIOD COMPARISONS
# =============================================================================
# Healthcare Parallel: WoW/MoM/YoY comparisons of operational metrics -
# equivalent to comparing current period vs prior period for hospital operations

print("\n" + "="*80)
print("CREATING: mff_gold.period_over_period")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TABLE mff_gold.period_over_period
    COMMENT 'Period-over-period metric comparisons (WoW, MoM, YoY) with trending indicators - equivalent to healthcare operational variance reports'
    AS
    
    WITH daily_metrics AS (
        SELECT
            DATE(o.created_at) as metric_date,
            COUNT(DISTINCT o.order_id) as daily_orders,
            SUM(o.price_usd) as daily_revenue,
            COUNT(DISTINCT o.order_id) * 1.0 / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as daily_cvr,
            SUM(CASE WHEN oir.order_item_refund_id IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT o.order_id), 0) as daily_refund_rate
        FROM mff_bronze.website_sessions ws
        LEFT JOIN mff_bronze.orders o 
            ON ws.website_session_id = o.website_session_id
        LEFT JOIN mff_bronze.order_items oi
            ON o.order_id = oi.order_id
        LEFT JOIN mff_bronze.order_item_refunds oir
            ON oi.order_item_id = oir.order_item_id
        WHERE DATE(o.created_at) IS NOT NULL
        GROUP BY DATE(o.created_at)
    )
    
    SELECT
        metric_date,
        
        -- Current metrics
        daily_orders,
        daily_revenue,
        ROUND(daily_cvr, 4) as daily_cvr,
        ROUND(daily_refund_rate, 4) as daily_refund_rate,
        
        -- Week-over-Week (WoW) Comparisons
        -- Healthcare: Weekly operational reviews by department directors
        LAG(daily_orders, 7) OVER (ORDER BY metric_date) as orders_7d_ago,
        ROUND(
            (daily_orders - LAG(daily_orders, 7) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_orders, 7) OVER (ORDER BY metric_date), 0),
            2
        ) as orders_wow_pct_change,
        
        LAG(daily_revenue, 7) OVER (ORDER BY metric_date) as revenue_7d_ago,
        ROUND(
            (daily_revenue - LAG(daily_revenue, 7) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_revenue, 7) OVER (ORDER BY metric_date), 0),
            2
        ) as revenue_wow_pct_change,
        
        -- Month-over-Month (MoM) Comparisons
        -- Healthcare: Monthly financial close and performance reviews
        LAG(daily_orders, 30) OVER (ORDER BY metric_date) as orders_30d_ago,
        ROUND(
            (daily_orders - LAG(daily_orders, 30) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_orders, 30) OVER (ORDER BY metric_date), 0),
            2
        ) as orders_mom_pct_change,
        
        LAG(daily_revenue, 30) OVER (ORDER BY metric_date) as revenue_30d_ago,
        ROUND(
            (daily_revenue - LAG(daily_revenue, 30) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_revenue, 30) OVER (ORDER BY metric_date), 0),
            2
        ) as revenue_mom_pct_change,
        
        -- Year-over-Year (YoY) Comparisons
        -- Healthcare: Strategic planning and board reporting
        LAG(daily_orders, 365) OVER (ORDER BY metric_date) as orders_365d_ago,
        ROUND(
            (daily_orders - LAG(daily_orders, 365) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_orders, 365) OVER (ORDER BY metric_date), 0),
            2
        ) as orders_yoy_pct_change,
        
        LAG(daily_revenue, 365) OVER (ORDER BY metric_date) as revenue_365d_ago,
        ROUND(
            (daily_revenue - LAG(daily_revenue, 365) OVER (ORDER BY metric_date)) * 100.0 / 
            NULLIF(LAG(daily_revenue, 365) OVER (ORDER BY metric_date), 0),
            2
        ) as revenue_yoy_pct_change,
        
        -- Trending Indicators
        -- Healthcare: Executive dashboard indicators (up arrow, down arrow, stable)
        CASE
            WHEN (daily_orders - LAG(daily_orders, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_orders, 30) OVER (ORDER BY metric_date), 0) > 5 THEN 'UP'
            WHEN (daily_orders - LAG(daily_orders, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_orders, 30) OVER (ORDER BY metric_date), 0) < -5 THEN 'DOWN'
            ELSE 'STABLE'
        END as orders_trend_30d,
        
        CASE
            WHEN (daily_revenue - LAG(daily_revenue, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_revenue, 30) OVER (ORDER BY metric_date), 0) > 5 THEN 'UP'
            WHEN (daily_revenue - LAG(daily_revenue, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_revenue, 30) OVER (ORDER BY metric_date), 0) < -5 THEN 'DOWN'
            ELSE 'STABLE'
        END as revenue_trend_30d,
        
        CASE
            WHEN (daily_cvr - LAG(daily_cvr, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_cvr, 30) OVER (ORDER BY metric_date), 0) > 5 THEN 'UP'
            WHEN (daily_cvr - LAG(daily_cvr, 30) OVER (ORDER BY metric_date)) * 100.0 / 
                 NULLIF(LAG(daily_cvr, 30) OVER (ORDER BY metric_date), 0) < -5 THEN 'DOWN'
            ELSE 'STABLE'
        END as cvr_trend_30d,
        
        CURRENT_TIMESTAMP() as created_at
        
    FROM daily_metrics
    ORDER BY metric_date
""")

# Verify table creation
result_count = spark.sql("SELECT COUNT(*) as cnt FROM mff_gold.period_over_period").collect()[0]['cnt']
print(f"\n✓ Created mff_gold.period_over_period with {result_count:,} rows")

# Show sample with most recent data
print("\nSample data (most recent 5 days):")
spark.sql("""
    SELECT 
        metric_date,
        daily_orders,
        orders_7d_ago,
        orders_wow_pct_change,
        orders_trend_30d,
        daily_revenue,
        revenue_wow_pct_change,
        revenue_trend_30d
    FROM mff_gold.period_over_period
    ORDER BY metric_date DESC
    LIMIT 5
""").show(truncate=False)

print("\nHealthcare Use Case:")
print("  • Weekly operational reviews: Are we up or down vs last week?")
print("  • Monthly financial variance reporting")
print("  • Strategic planning: YoY growth trends")
print("  • Executive dashboards: Trending indicators (UP/DOWN/STABLE)")

In [0]:
# =============================================================================
# TABLE 3: EXECUTIVE SCORECARD
# =============================================================================
# Healthcare Parallel: Balanced scorecard with composite performance score -
# equivalent to hospital quality dashboard, CMS Star Ratings, or Leapfrog scores

print("\n" + "="*80)
print("CREATING: mff_gold.executive_scorecard")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TABLE mff_gold.executive_scorecard
    COMMENT 'Monthly executive scorecard with composite performance score (0-100) and RAG status - equivalent to healthcare balanced scorecard'
    AS
    
    WITH monthly_metrics AS (
        -- Calculate monthly metrics
        SELECT
            DATE_TRUNC('MONTH', o.created_at) as score_month,
            COUNT(DISTINCT o.order_id) as monthly_orders,
            SUM(o.price_usd) as monthly_revenue,
            SUM(o.price_usd - (oi.price_usd * oi.cogs_usd)) as monthly_margin,
            COUNT(DISTINCT o.order_id) * 1.0 / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as monthly_cvr,
            SUM(CASE WHEN oir.order_item_refund_id IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT oi.order_item_id), 0) as monthly_refund_rate,
            -- Cross-sell rate: orders with 2+ items / total orders
            SUM(CASE WHEN o.items_purchased > 1 THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT o.order_id), 0) as monthly_cross_sell_rate
        FROM mff_bronze.website_sessions ws
        LEFT JOIN mff_bronze.orders o 
            ON ws.website_session_id = o.website_session_id
        LEFT JOIN mff_bronze.order_items oi
            ON o.order_id = oi.order_id
        LEFT JOIN mff_bronze.order_item_refunds oir
            ON oi.order_item_id = oir.order_item_id
        WHERE o.created_at IS NOT NULL
        GROUP BY DATE_TRUNC('MONTH', o.created_at)
    ),
    
    monthly_with_targets AS (
        SELECT
            score_month,
            monthly_orders,
            monthly_revenue,
            monthly_margin,
            monthly_cvr,
            monthly_refund_rate,
            monthly_cross_sell_rate,
            
            -- Calculate prior month for MoM growth
            LAG(monthly_revenue) OVER (ORDER BY score_month) as prior_month_revenue,
            
            -- Calculate 90-day averages as benchmarks
            -- Healthcare: Compare to rolling baseline (peer benchmarks, historical performance)
            AVG(monthly_cvr) OVER (
                ORDER BY score_month
                ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ) as cvr_90d_avg,
            
            -- Define targets (in healthcare: these would be CMS targets, HEDIS targets, internal goals)
            0.05 as refund_rate_target,  -- Target: < 5% refund rate
            0.08 as cross_sell_target    -- Target: > 8% cross-sell rate
            
        FROM monthly_metrics
    ),
    
    scored_metrics AS (
        SELECT
            score_month,
            monthly_orders,
            monthly_revenue,
            ROUND(monthly_cvr, 4) as monthly_cvr,
            ROUND(monthly_refund_rate, 4) as monthly_refund_rate,
            ROUND(monthly_cross_sell_rate, 4) as monthly_cross_sell_rate,
            
            -- Component Scores (0-100 for each dimension)
            -- Healthcare: Similar to scoring HEDIS measures or quality indicators
            
            -- 1. Revenue Growth Score (25% weight)
            CASE
                WHEN prior_month_revenue IS NULL THEN 50.0  -- First month: neutral score
                WHEN (monthly_revenue - prior_month_revenue) / NULLIF(prior_month_revenue, 0) > 0.10 THEN 100.0  -- >10% growth: excellent
                WHEN (monthly_revenue - prior_month_revenue) / NULLIF(prior_month_revenue, 0) > 0.05 THEN 85.0   -- >5% growth: good
                WHEN (monthly_revenue - prior_month_revenue) / NULLIF(prior_month_revenue, 0) > 0 THEN 70.0      -- positive growth: acceptable
                WHEN (monthly_revenue - prior_month_revenue) / NULLIF(prior_month_revenue, 0) > -0.05 THEN 50.0  -- slight decline: concerning
                ELSE 25.0  -- >5% decline: poor
            END as revenue_growth_score,
            
            -- 2. CVR Performance Score (25% weight)
            CASE
                WHEN monthly_cvr > cvr_90d_avg * 1.10 THEN 100.0  -- 10% above average: excellent
                WHEN monthly_cvr > cvr_90d_avg * 1.05 THEN 85.0   -- 5% above average: good
                WHEN monthly_cvr >= cvr_90d_avg * 0.95 THEN 70.0  -- Within 5% of average: acceptable
                WHEN monthly_cvr >= cvr_90d_avg * 0.90 THEN 50.0  -- 5-10% below average: concerning
                ELSE 25.0  -- >10% below average: poor
            END as cvr_performance_score,
            
            -- 3. Refund Rate Score (25% weight) - lower is better
            CASE
                WHEN monthly_refund_rate < refund_rate_target * 0.5 THEN 100.0   -- <2.5%: excellent
                WHEN monthly_refund_rate < refund_rate_target * 0.75 THEN 85.0   -- <3.75%: good
                WHEN monthly_refund_rate <= refund_rate_target THEN 70.0         -- <5%: acceptable
                WHEN monthly_refund_rate <= refund_rate_target * 1.25 THEN 50.0  -- <6.25%: concerning
                ELSE 25.0  -- >6.25%: poor
            END as refund_rate_score,
            
            -- 4. Cross-Sell Rate Score (25% weight)
            CASE
                WHEN monthly_cross_sell_rate > cross_sell_target * 1.25 THEN 100.0  -- >10%: excellent
                WHEN monthly_cross_sell_rate > cross_sell_target * 1.10 THEN 85.0   -- >8.8%: good
                WHEN monthly_cross_sell_rate >= cross_sell_target THEN 70.0         -- >8%: acceptable
                WHEN monthly_cross_sell_rate >= cross_sell_target * 0.75 THEN 50.0  -- >6%: concerning
                ELSE 25.0  -- <6%: poor
            END as cross_sell_rate_score
            
        FROM monthly_with_targets
    )
    
    SELECT
        score_month,
        monthly_orders,
        monthly_revenue,
        monthly_cvr,
        monthly_refund_rate,
        monthly_cross_sell_rate,
        
        -- Individual component scores
        ROUND(revenue_growth_score, 2) as revenue_growth_score,
        ROUND(cvr_performance_score, 2) as cvr_performance_score,
        ROUND(refund_rate_score, 2) as refund_rate_score,
        ROUND(cross_sell_rate_score, 2) as cross_sell_rate_score,
        
        -- Composite Performance Score (weighted average)
        -- Healthcare: Overall quality score, CMS Star Rating (1-5 converted to 0-100)
        ROUND(
            (revenue_growth_score * 0.25 + 
             cvr_performance_score * 0.25 + 
             refund_rate_score * 0.25 + 
             cross_sell_rate_score * 0.25),
            2
        ) as composite_performance_score,
        
        -- RAG Status (Red/Amber/Green)
        -- Healthcare: Traffic light indicators for executive dashboards
        CASE
            WHEN (revenue_growth_score * 0.25 + cvr_performance_score * 0.25 + 
                  refund_rate_score * 0.25 + cross_sell_rate_score * 0.25) >= 80 THEN 'GREEN'
            WHEN (revenue_growth_score * 0.25 + cvr_performance_score * 0.25 + 
                  refund_rate_score * 0.25 + cross_sell_rate_score * 0.25) >= 60 THEN 'AMBER'
            ELSE 'RED'
        END as rag_status,
        
        CURRENT_TIMESTAMP() as created_at
        
    FROM scored_metrics
    ORDER BY score_month
""")

# Verify table creation
result_count = spark.sql("SELECT COUNT(*) as cnt FROM mff_gold.executive_scorecard").collect()[0]['cnt']
print(f"\n✓ Created mff_gold.executive_scorecard with {result_count:,} rows")

# Show sample
print("\nSample scorecard (most recent 6 months):")
spark.sql("""
    SELECT 
        DATE_FORMAT(score_month, 'yyyy-MM') as month,
        composite_performance_score,
        rag_status,
        revenue_growth_score,
        cvr_performance_score,
        refund_rate_score,
        cross_sell_rate_score,
        monthly_revenue
    FROM mff_gold.executive_scorecard
    ORDER BY score_month DESC
    LIMIT 6
""").show(truncate=False)

print("\nScoring Methodology:")
print("  • 4 dimensions, each weighted 25%")
print("  • 100 = Excellent, 85 = Good, 70 = Acceptable, 50 = Concerning, 25 = Poor")
print("  • RAG Status: GREEN (80+), AMBER (60-79), RED (<60)")
print("\nHealthcare Equivalent:")
print("  • CMS Star Ratings (1-5 stars)")
print("  • Hospital quality scorecards")
print("  • Balanced scorecard methodology")
print("  • HEDIS composite measure scoring")

In [0]:
# =============================================================================
# TABLE 4: CHANNEL EFFICIENCY MATRIX
# =============================================================================
# Healthcare Parallel: Payer mix analysis, referral source efficiency -
# which insurance types or referring physicians drive most profitable patients

print("\n" + "="*80)
print("CREATING: mff_gold.channel_efficiency_matrix")
print("="*80)

spark.sql("""
    CREATE OR REPLACE TABLE mff_gold.channel_efficiency_matrix
    COMMENT 'Channel efficiency metrics with multi-dimensional rankings - equivalent to healthcare payer mix and referral source efficiency analysis'
    AS
    
    WITH channel_metrics AS (
        -- Calculate comprehensive metrics per channel
        -- Healthcare: Calculate per insurance type or referring physician
        SELECT
            COALESCE(ws.utm_source, 'direct') as channel,
            
            -- Volume metrics
            COUNT(DISTINCT ws.website_session_id) as total_sessions,
            COUNT(DISTINCT o.order_id) as total_orders,
            
            -- Efficiency metrics
            COUNT(DISTINCT o.order_id) * 1.0 / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as cvr,
            SUM(o.price_usd) / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as revenue_per_session,
            SUM(o.price_usd - (oi.price_usd * oi.cogs_usd)) / NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as margin_per_session,
            
            -- Quality metrics
            SUM(CASE WHEN oir.order_item_refund_id IS NOT NULL THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT oi.order_item_id), 0) as refund_rate,
            
            -- Loyalty metrics
            SUM(CASE WHEN ws.is_repeat_session = 1 THEN 1 ELSE 0 END) * 1.0 / 
                NULLIF(COUNT(DISTINCT ws.website_session_id), 0) as repeat_customer_rate,
            
            -- LTV metric (for customers acquired via this channel)
            AVG(u.gross_lifetime_value) as avg_ltv_acquired
            
        FROM mff_bronze.website_sessions ws
        LEFT JOIN mff_bronze.orders o
            ON ws.website_session_id = o.website_session_id
        LEFT JOIN mff_bronze.order_items oi
            ON o.order_id = oi.order_id
        LEFT JOIN mff_bronze.order_item_refunds oir
            ON oi.order_item_id = oir.order_item_id
        LEFT JOIN (
            -- Calculate LTV per user at first touch channel
            SELECT 
                user_id,
                MIN(utm_source) as first_channel,
                SUM(price_usd) as gross_lifetime_value
            FROM mff_bronze.website_sessions ws2
            LEFT JOIN mff_bronze.orders o2 ON ws2.website_session_id = o2.website_session_id
            GROUP BY user_id
        ) u ON ws.user_id = u.user_id AND COALESCE(ws.utm_source, 'direct') = COALESCE(u.first_channel, 'direct')
        
        GROUP BY COALESCE(ws.utm_source, 'direct')
    ),
    
    ranked_channels AS (
        SELECT
            channel,
            total_sessions,
            total_orders,
            ROUND(cvr, 4) as cvr,
            ROUND(revenue_per_session, 2) as revenue_per_session,
            ROUND(margin_per_session, 2) as margin_per_session,
            ROUND(refund_rate, 4) as refund_rate,
            ROUND(repeat_customer_rate, 4) as repeat_customer_rate,
            ROUND(avg_ltv_acquired, 2) as avg_ltv_acquired,
            
            -- Rankings on each dimension (1 = best)
            -- Healthcare: Rank payers by profitability, quality, volume
            DENSE_RANK() OVER (ORDER BY revenue_per_session DESC) as revenue_per_session_rank,
            DENSE_RANK() OVER (ORDER BY margin_per_session DESC) as margin_per_session_rank,
            DENSE_RANK() OVER (ORDER BY refund_rate ASC) as refund_rate_rank,  -- Lower is better
            DENSE_RANK() OVER (ORDER BY repeat_customer_rate DESC) as repeat_rate_rank,
            DENSE_RANK() OVER (ORDER BY avg_ltv_acquired DESC) as ltv_rank
            
        FROM channel_metrics
        WHERE total_sessions > 100  -- Filter out channels with low volume
    )
    
    SELECT
        channel,
        total_sessions,
        total_orders,
        cvr,
        revenue_per_session,
        margin_per_session,
        refund_rate,
        repeat_customer_rate,
        avg_ltv_acquired,
        
        -- Individual dimension ranks
        revenue_per_session_rank,
        margin_per_session_rank,
        refund_rate_rank,
        repeat_rate_rank,
        ltv_rank,
        
        -- Composite Efficiency Rank
        -- Healthcare: Overall payer efficiency score combining multiple dimensions
        -- Lower is better (rank 1 = most efficient)
        ROUND(
            (revenue_per_session_rank * 0.25 +
             margin_per_session_rank * 0.25 +
             refund_rate_rank * 0.20 +
             repeat_rate_rank * 0.15 +
             ltv_rank * 0.15),
            2
        ) as composite_efficiency_rank,
        
        -- Efficiency Tier Assignment
        -- Healthcare: Tier 1 payers (high value), Tier 2 (standard), Tier 3 (low value)
        CASE
            WHEN DENSE_RANK() OVER (
                ORDER BY (
                    revenue_per_session_rank * 0.25 +
                    margin_per_session_rank * 0.25 +
                    refund_rate_rank * 0.20 +
                    repeat_rate_rank * 0.15 +
                    ltv_rank * 0.15
                )
            ) <= 2 THEN 'TIER_1_HIGH_EFFICIENCY'
            WHEN DENSE_RANK() OVER (
                ORDER BY (
                    revenue_per_session_rank * 0.25 +
                    margin_per_session_rank * 0.25 +
                    refund_rate_rank * 0.20 +
                    repeat_rate_rank * 0.15 +
                    ltv_rank * 0.15
                )
            ) <= 5 THEN 'TIER_2_STANDARD'
            ELSE 'TIER_3_LOW_EFFICIENCY'
        END as efficiency_tier,
        
        CURRENT_TIMESTAMP() as created_at
        
    FROM ranked_channels
    ORDER BY composite_efficiency_rank
""")

# Verify table creation
result_count = spark.sql("SELECT COUNT(*) as cnt FROM mff_gold.channel_efficiency_matrix").collect()[0]['cnt']
print(f"\n✓ Created mff_gold.channel_efficiency_matrix with {result_count:,} rows")

# Show full efficiency matrix
print("\nChannel Efficiency Matrix (ranked by composite efficiency):")
spark.sql("""
    SELECT 
        channel,
        total_sessions,
        revenue_per_session,
        margin_per_session,
        cvr,
        avg_ltv_acquired,
        composite_efficiency_rank,
        efficiency_tier
    FROM mff_gold.channel_efficiency_matrix
    ORDER BY composite_efficiency_rank
""").show(20, truncate=False)

print("\nRanking Methodology:")
print("  • Revenue per session: 25% weight")
print("  • Margin per session: 25% weight")
print("  • Refund rate (inverse): 20% weight")
print("  • Repeat customer rate: 15% weight")
print("  • Average LTV acquired: 15% weight")
print("\nHealthcare Equivalent:")
print("  • Payer mix analysis: Which insurance types are most profitable?")
print("  • Referral source analysis: Which physicians send best patients?")
print("  • Service line performance: Which specialties drive margin?")
print("  • Clinic efficiency: Which locations perform best?")
print("\nBusiness Decisions Enabled:")
print("  • Marketing budget allocation: Invest more in Tier 1 channels")
print("  • Channel optimization: Improve or exit Tier 3 channels")
print("  • Acquisition strategy: Focus on high-LTV channels")
print("  • Resource allocation: Staff/support proportional to efficiency")